In [ ]:
## 1 set up environment

# Import required libraries for web scraping
import requests
from bs4 import BeautifulSoup

print("✅ Libraries imported successfully")

📚 Setting up the environment...
✅ Libraries imported successfully


In [3]:
### request with proper headers

# Now let's try with proper headers
print("🌐 Attempting to access webpage with proper headers...")

url = "https://www.foxnews.com/world/lavrov-warns-europe-retaliation-zelenskyy-opens-reconstruction-talks-trump-officials"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) "
    + " Chrome/39.0.2171.95 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers)
    print(f"📡 Response status code: {response.status_code}")

    if response.status_code != 200:
        raise Exception(f"Failed to fetch webpage: Status code {response.status_code}")

    content_length = len(response.content)
    print(f"✅ Successfully retrieved the webpage! Received {content_length} bytes of data.")
except Exception as e:
    print(f"❌ Error: {str(e)}")

🌐 Attempting to access webpage with proper headers...
📡 Response status code: 200
✅ Successfully retrieved the webpage! Received 362362 bytes of data.


In [6]:
# Parse the HTML and extract content
print("🔍 Parsing webpage content...")

try:
    # Parse the HTML
    webpage = BeautifulSoup(response.content, "html.parser")

    # Extract title
    title = webpage.title.string.strip()
    print("\n📑 Page Title:")
    print("-" * 40)
    print(title)
    print("-" * 40)

    # Extract paragraphs
    print("\n📝 Article Content:")
    print("-" * 40)

    
    description_html = webpage.select(".col-lg-8")  # <----------------- !!! Our selector !!!
    texts = [text.get_text().strip() for text in description_html]
    text = "\n".join(texts)
    print(text)
    print("-" * 40)

    print("\n✅ Content extracted successfully")
except Exception as e:
    print(f"❌ Error parsing content: {str(e)}")

🔍 Parsing webpage content...

📑 Page Title:
----------------------------------------
Lavrov warns Russia will retaliate if Europe deploys troops to Ukraine | Fox News
----------------------------------------

📝 Article Content:
----------------------------------------

----------------------------------------

✅ Content extracted successfully


In [6]:
#webpage.keys()
# print(webpage[:500])
webpage

<!DOCTYPE html>

<html data-n-head="%7B%22lang%22:%7B%22ssr%22:%22en%22%7D%7D" data-n-head-ssr="" lang="en">
<head>
<title>Lavrov warns Russia will retaliate if Europe deploys troops to Ukraine | Fox News</title><meta content="IE=edge,chrome=1" data-n-head="ssr" http-equiv="X-UA-Compatible"/><meta content="text/html; charset=utf-8" data-hid="content-type" data-n-head="ssr" http-equiv="content-type"/><meta content="on" data-hid="x-dns-prefetch-control" data-n-head="ssr" http-equiv="x-dns-prefetch-control"/><meta charset="utf-8" data-n-head="ssr"/><meta content="width=device-width, minimum-scale=1.0, initial-scale=1.0" data-hid="viewport" data-n-head="ssr" name="viewport"/><meta content="//static.foxnews.com/static/orion/styles/img/fox-news/favicons/mstile-70x70.png" data-n-head="ssr" name="msapplication-square70x70logo"/><meta content="//static.foxnews.com/static/orion/styles/img/fox-news/favicons/mstile-150x150.png" data-n-head="ssr" name="msapplication-square150x150logo"/><meta conten

In [37]:
#webpage.select("body")
print(webpage.select("p"))

[<p class="more-copyright">
        This material may not be published, broadcast, rewritten, or redistributed. ©2025 FOX News Network, LLC. All rights reserved. Quotes displayed in real-time or delayed by at least 15 minutes. Market data provided by <a data-omtr-intcmp="topnav_more_factset" href="https://www.factset.com/">Factset</a>. Powered and implemented by <a data-omtr-intcmp="topnav_more_factset_digital_solutions" href="https://www.factset.com/solutions/business-needs/digital-solutions">FactSet Digital Solutions</a>. <a data-omtr-intcmp="topnav_more_factset_privacy" href="https://www.factset.com/privacy">Legal Statement</a>. Mutual Fund and ETF data provided by <a data-omtr-intcmp="topnav_more_refinitive_info" href="https://lipperalpha.refinitiv.com/">Refinitiv Lipper</a>.
      </p>, <p data-v-324c3b57="">U.S. Ambassador to NATO Matt Whitaker joined 'Fox &amp; Friends First' to discuss Russia's overnight attack on Ukraine as Zelenskyy is set to meet with European leaders in Lon

In [36]:
#for tag in webpage.find_all():
#    print(tag.name)

seen = set()

for tag in webpage.find_all():
    name = tag.name
    if name not in seen:
        seen.add(name)

print(seen)

{'hedgehog-reactions', 'html', 'link', 'picture', 'footer', 'script', 'a', 'meta', 'time', 'section', 'style', 'source', 'form', 'head', 'h4', 'h2', 'br', 'p', 'body', 'strong', 'h3', 'article', 'button', 'span', 'div', 'input', 'h1', 'header', 'title', 'main', 'nav', 'ul', 'li', 'i', 'img', 'fieldset'}


In [ ]:
### The Key Improvement: Focus Your Search ###

# 1. Find the parent container that holds the main article text
# We use .find() because there should only be one main article body.
article_container = webpage.find("div", class_="article-body")

if article_container:
    # 2. Find all paragraph tags *ONLY* within this container
    # This automatically excludes all paragraphs in the header, footer, etc.
    paragraphs = article_container.find_all("p")

    # 3. Clean up the text:
    article_text_lines = []
    
    for p in paragraphs:
        text = p.get_text().strip()
        
        # Optional: Add a check to exclude unwanted lines often found inside the container
        # These checks filter out things like image captions or related links that often
        # appear in a <p> tag but aren't part of the core narrative.
        
        # Check 1: Skip if the text is short (like an empty line or just a link title)
        if len(text) < 20: 
             continue
        
        # Check 2: Skip elements with specific unwanted classes or content patterns
        # Example: Skip paragraphs that are clearly image captions
        if p.get('class') and 'speakable' not in p.get('class'): # keep 'speakable' paragraphs
            # You would need to inspect the HTML to find which paragraphs to skip (e.g., image captions)
            # For simplicity, we will rely on the container selection for now, 
            # but keep this option in mind for further refinement.
            pass # Keep all paragraphs for now, the container does most of the work.

        article_text_lines.append(text)
        
    # 4. Join the cleaned lines into one block of text
    article_text = "\n\n".join(article_text_lines)

    print("--- Clean Extracted Article Text (First 1500 characters) ---\n")
    print(article_text[:1500] + "...")
    
else:
    print("Article body container not found. Check the class name 'article-body' or the website structure.")

--- Clean Extracted Article Text (First 1500 characters) ---

U.S. Ambassador to NATO Matt Whitaker joined 'Fox & Friends First' to discuss Russia's overnight attack on Ukraine as Zelenskyy is set to meet with European leaders in London.

Russian Foreign Minister Sergey Lavrov warned on Wednesday that Moscow will retaliate if European governments deploy troops to Ukraine or seize frozen Russian assets to support Kyiv, according to Reuters.

Lavrov delivered the remarks before the Federation Council, Russia’s upper house of parliament, outlining Moscow’s stance on the war and its clash with the West. Reuters reported that Lavrov insisted Russia does not seek war with Europe but is prepared to act if it views Western countries as escalating the conflict.

"We will respond to any hostile steps, including the deployment of European military contingents in Ukraine and the expropriation of Russian assets. And we are already prepared for this response," Lavrov said, according to Reuters.

Lav